# Key custody and handle refusals

Real WebCrypto non-extractable keys, extractable-pair refusal and local cryptographic custody using DER signatures converted to raw P-256. This does not call or simulate a KMS service.

Open this file in JupyterLab or VS Code with the **Deno** kernel. From a repository checkout, install Deno 2.5.6, Python 3.13.9 and uv 0.9.9 (the `.tool-versions` pins), run `uv sync --frozen` and `uv run --frozen deno jupyter --install`, then select Deno and Run All. Any Jupyter-capable editor can use that installed kernel. See [launch and automated checks](README.md).

The saved notebook installs exact published packages. `pnpm notebooks:local` checks the packed candidate; `pnpm notebooks:published` runs these cells unchanged against npm. Fixed times describe a demonstration, not current credentials. Keys and signatures are generated, never golden output. Host authorization, replay and revocation stay outside this notebook.


In [1]:
import signerMetadata from "npm:@bounded-authority-protocol/signer@2.0.0/package.json" with { type: "json" };
import verifierMetadata from "npm:@bounded-authority-protocol/verifier@2.0.0/package.json" with { type: "json" };
if (signerMetadata.version !== "2.0.0" || verifierMetadata.version !== "2.0.0") throw new Error("package version mismatch");
console.log("exact signer and verifier releases loaded");

exact signer and verifier releases loaded


In [2]:
import { Buffer } from "node:buffer";
import { generateKeyPairSync, sign } from "node:crypto";
import { jwkFromPublicKey, thumbprint, v3 } from "npm:@bounded-authority-protocol/verifier@2.0.0";
import type { Es256KeyHandle, KeyHandle, KeyRole } from "npm:@bounded-authority-protocol/signer@2.0.0";

const fromBase64url = (text: string): Uint8Array => new Uint8Array(Buffer.from(text, "base64url"));

/** An Ed25519 handle over a fresh in-process key (contract-majors 1 and 2). */
function testKeyHandle(keyId: string, role: KeyRole = "holder"): KeyHandle {
  const { publicKey, privateKey } = generateKeyPairSync("ed25519");
  const raw = fromBase64url(publicKey.export({ format: "jwk" }).x!);
  return {
    sign: (message) => new Uint8Array(sign(null, message, privateKey)),
    publicKey: () => raw,
    thumbprint: () => thumbprint(jwkFromPublicKey(raw)),
    keyIdentity: () => ({ keyId, publicKey: raw }),
    signingIdentity: () => ({ role, keyId, publicKey: raw }),
  };
}

/** An ES256 handle over a fresh in-process P-256 key (contract-major 3). */
function testEs256KeyHandle(keyId: string, role: KeyRole = "holder"): Es256KeyHandle {
  const { publicKey, privateKey } = generateKeyPairSync("ec", { namedCurve: "P-256" });
  const { x, y } = publicKey.export({ format: "jwk" }) as { x: string; y: string };
  const raw = new Uint8Array([0x04, ...fromBase64url(x), ...fromBase64url(y)]);
  return {
    sign: (message) => new Uint8Array(sign("sha256", message, { key: privateKey, dsaEncoding: "ieee-p1363" })),
    publicKey: () => raw,
    thumbprint: () => v3.thumbprint({ crv: "P-256", kty: "EC", x, y }),
    keyIdentity: () => ({ keyId, publicKey: raw }),
    signingIdentity: () => ({ role, keyId, publicKey: raw }),
  };
}
console.log("demonstration key constructors ready");

demonstration key constructors ready


In [3]:
import { ed25519KeyHandle, es256KeyHandle, signGrant, signV3Anchor } from "npm:@bounded-authority-protocol/signer@2.0.0";

{

// WebKit for Linux can transiently refuse key generation; retry only OperationError.
async function generatePair(algorithm: Algorithm | EcKeyGenParams, extractable = false): Promise<CryptoKeyPair> {
  for (let attempt = 0; attempt < 3; attempt++) {
    try { return await crypto.subtle.generateKey(algorithm, extractable, ["sign", "verify"]) as CryptoKeyPair; }
    catch (error) { if ((error as { name?: string }).name !== "OperationError" || attempt === 2) throw error; }
  }
  throw new Error("key generation failed");
}

// You create the pair and own its lifetime; the library never generates, imports or exports a private key.
const issuerPair = await generatePair({ name: "Ed25519" });
const issuer = await ed25519KeyHandle(issuerPair, { keyId: "issuer-key-1", role: "issuer" });
if (!issuer.ok) throw new Error(issuer.error);

const agentPair = await generatePair({ name: "Ed25519" });
const agent = await ed25519KeyHandle(agentPair); // role "holder"
if (!agent.ok) throw new Error(agent.error);

const grant = await signGrant({
  issuer: "https://issuer.example", grantId: "urn:example:grant:calendar",
  audiences: ["https://calendar.example"],
  issuedAt: 1_767_225_600, notBefore: 1_767_225_600, expiresAt: 1_767_232_800,
  holderThumbprint: await agent.value.thumbprint(),
  operations: [{ name: "create_event", selectors: [{ kind: "all" }] }],
}, issuer.value);

// Contract-major 3 (ES256): the same with an ECDSA P-256 pair.
const ledgerPair = await generatePair({ name: "ECDSA", namedCurve: "P-256" });
const ledger = await es256KeyHandle(ledgerPair, { keyId: "ledger-key-1" });
if (!ledger.ok) throw new Error(ledger.error);
const anchor = await signV3Anchor(
  { anchorId: "urn:example:anchor:0", chainId: "urn:example:chain:ledger", sequence: 0, chainHash: new Uint8Array(32) },
  ledger.value,
);

// An extractable private key is refused.
const exportable = await generatePair({ name: "Ed25519" }, true);
const refused = await ed25519KeyHandle(exportable);
console.log(`grant: ${grant.ok}; anchor: ${anchor.ok}; extractable pair: ${refused.ok ? "accepted" : refused.error}`);
if (!grant.ok || !anchor.ok || refused.ok || refused.error !== "invalid_key_handle") throw new Error("factory contract");
}

grant: true; anchor: true; extractable pair: invalid_key_handle


In [4]:
import { generateKeyPairSync, randomUUID, sign } from "node:crypto";
import { signV3Grant, signV3Report, type Es256KeyHandle, type KeyRole } from "npm:@bounded-authority-protocol/signer@2.0.0";
import { base64urlEncode, strUtf8, utf8Str, v3, type Tagged } from "npm:@bounded-authority-protocol/verifier@2.0.0";


/** ASN.1 DER `SEQUENCE { INTEGER r, INTEGER s }` to the 64-byte raw `r || s` of a P-256 signature. */
function derToRawP256(der: Uint8Array): Uint8Array {
  const malformed = () => new Error("not a DER-encoded P-256 ECDSA signature");
  if (der[0] !== 0x30 || der[1] !== der.length - 2) throw malformed();
  const raw = new Uint8Array(64);
  let at = 2;
  for (const offset of [0, 32]) {
    const length = der[at + 1] ?? 0;
    if (der[at] !== 0x02 || length < 1 || length > 33 || at + 2 + length > der.length) throw malformed();
    let value = der.subarray(at + 2, at + 2 + length);
    if ((value[0]! & 0x80) !== 0 || (value.length > 1 && value[0] === 0 && (value[1]! & 0x80) === 0)) throw malformed();
    if (value.length === 33) {
      if (value[0] !== 0x00) throw malformed();
      value = value.subarray(1); // the sign byte DER adds when the top bit is set
    }
    raw.set(value, offset + 32 - value.length); // DER drops leading zero bytes; restore them
    at += 2 + length;
  }
  if (at !== der.length) throw malformed();
  return raw;
}

/** An ES256 handle over a custodian that signs with ECDSA P-256 SHA-256 and returns DER. */
async function kmsEs256KeyHandle(
  signDer: (message: Uint8Array) => Promise<Uint8Array>, // your KMS call
  spki: Uint8Array,                                      // the key's DER SubjectPublicKeyInfo
  keyId: string,
  role: KeyRole,
): Promise<Es256KeyHandle> {
  const key = await crypto.subtle.importKey("spki", new Uint8Array(spki), { name: "ECDSA", namedCurve: "P-256" }, true, ["verify"]);
  const publicKey = new Uint8Array(await crypto.subtle.exportKey("raw", key)); // 0x04 || x || y
  const b64 = (bytes: Uint8Array) => utf8Str(base64urlEncode(bytes));
  const jkt = v3.thumbprint({ crv: "P-256", kty: "EC", x: b64(publicKey.subarray(1, 33)), y: b64(publicKey.subarray(33)) });
  return {
    sign: async (message) => derToRawP256(await signDer(message)),
    publicKey: () => publicKey,
    thumbprint: () => jkt,
    keyIdentity: () => ({ keyId, publicKey }),
    signingIdentity: () => ({ role, keyId, publicKey }),
  };
}

// A local cryptographic custodian: a real P-256 key returning DER; this does not call a KMS service.
const custodian = generateKeyPairSync("ec", { namedCurve: "P-256" });
const signDer = async (message: Uint8Array) => new Uint8Array(sign("sha256", message, custodian.privateKey));
const spki = new Uint8Array(custodian.publicKey.export({ type: "spki", format: "der" }));

const issuer = await kmsEs256KeyHandle(signDer, spki, "backup-issuer-kms-1", "issuer");
const agent = testEs256KeyHandle("backup-agent-1");
let verified = 0;
for (let i = 0; i < 16; i++) { // about half of the custodian's signatures have a high s
  const grant = await signV3Grant({
    issuer: "https://backup.example", grantId: `urn:example:grant:snapshot-${i}`,
    audiences: ["https://snapshots.example"],
    issuedAt: 1_767_225_600, notBefore: 1_767_225_600, expiresAt: 1_767_229_200,
    holderThumbprint: await agent.thumbprint(),
    operations: [{ name: "create_snapshot", selectors: [{ kind: "equals", path: ["volume"], value: { t: "string", v: strUtf8("data-1") } }] }],
  }, issuer);
  if (!grant.ok) throw new Error(`signV3Grant: ${grant.error}`);
  const request = {
    method: "POST", targetUri: "https://snapshots.example/snapshots", invocationId: randomUUID(), operation: "create_snapshot",
    castArguments: { t: "object", v: new Map<string, Tagged>([["volume", { t: "string", v: strUtf8("data-1") }]]) } satisfies Tagged,
  };
  const envelope = await signV3Report({ grantCompact: grant.value.grant, ...request }, agent, { issuedAt: 1_767_225_660 });
  if (!envelope.ok) throw new Error(`signV3Report: ${envelope.error}`);
  const facts = v3.checkEnvelope(envelope.value.grant, envelope.value.proof, {
    ...request,
    trustedIssuer: { keyId: "backup-issuer-kms-1", publicKey: await issuer.publicKey() },
    issuer: "https://backup.example", audience: "https://snapshots.example",
    evaluationTime: 1_767_225_700, clockSkew: 60, proofMaxAge: 300, nonce: { kind: "not_required" },
  });
  if (facts.ok) verified++;
}
console.log(`${verified} of 16 envelopes verify under the custodian's key`);
if (verified !== 16) throw new Error("custody encoding verification");

16 of 16 envelopes verify under the custodian's key


In [5]:
const wrongEncoding = { ...issuer, sign: signDer };
const encodingRefusal = await signV3Anchor({ anchorId: "urn:example:anchor:encoding", chainId: "urn:example:chain:1", sequence: 0, chainHash: new Uint8Array(32) }, wrongEncoding, { anchoredAt: 1500 });
if (encodingRefusal.ok || encodingRefusal.error !== "signing_failed") throw new Error("DER accepted as raw signature");
console.log("unconverted DER signature refused");

unconverted DER signature refused


## Cleanup

These cells keep keys and compacts only in the kernel. Shut down the kernel to discard them. No private material is persisted. The runner shuts down the kernel and removes its consumer directory, package caches and temporary kernel registration, and fails on files left in the notebook working directory.

Continue with the [guides](../guides/getting-started.md).
